# Test LM Optimizer with Synthetic Data

This notebook creates synthetic landmarks and camera poses to test the LM optimizer:
- Generates ground truth camera trajectory and 3D landmarks
- Adds noise to simulate realistic measurements
- Feeds data to LM optimizer and tracks optimization results
- Visualizes ground truth, initial state, and optimized results
- Provides analysis plots for debugging


In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation as R
from easydict import EasyDict as edict
import gtsam

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from point2pose.modules.optimizer.lm_optimizer import LMGraphOptimizer
from point2pose.data_types.object_frame_data import ObjectFrameData
from point2pose.utils.transform import inverse_SE3, transform_pts

# Use interactive backend for rotatable 3D plots
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('matplotlib', 'widget')
except:
    try:
        if ipython is not None:
            ipython.run_line_magic('matplotlib', 'notebook')
    except:
        import matplotlib
        matplotlib.use('inline')
        print("Note: Interactive 3D plots not available. Using static plots.")

plt.ion()  # Interactive mode


In [ ]:
# Configuration
np.random.seed(42)  # For reproducibility

# Simulation parameters
NUM_FRAMES = 10
NUM_LANDMARKS = 20
OBJ_ID = 0

# Noise parameters
POSE_NOISE_TRANSLATION = 0.05  # meters
POSE_NOISE_ROTATION = 0.05  # radians
LANDMARK_MEASUREMENT_NOISE = 0.00  # meters (for range)
BEARING_NOISE = 0.00  # radians (for bearing)

# Trajectory parameters
TRAJECTORY_RADIUS = 2.0  # meters
TRAJECTORY_HEIGHT = 1.0  # meters
LANDMARK_SPREAD = 3.0  # meters

print(f"Configuration:")
print(f"  Frames: {NUM_FRAMES}")
print(f"  Landmarks: {NUM_LANDMARKS}")
print(f"  Pose noise (translation): {POSE_NOISE_TRANSLATION} m")
print(f"  Pose noise (rotation): {POSE_NOISE_ROTATION} rad")
print(f"  Measurement noise: {LANDMARK_MEASUREMENT_NOISE} m")


In [ ]:
def create_ground_truth_trajectory(num_frames, radius, height):
    """Create a circular trajectory around the origin."""
    poses_gt = []
    
    for i in range(num_frames):
        angle = 2 * np.pi * i / num_frames
        
        # Position: circular path
        x = radius * np.cos(angle)
        y = radius * np.sin(angle)
        z = height
        
        # Orientation: camera looks toward origin
        # Camera z-axis points forward, y-axis down
        forward = np.array([-x, -y, -z])
        forward_norm = np.linalg.norm(forward)
        if forward_norm < 1e-9:
            forward = np.array([0, 0, -1])
        else:
            forward = forward / forward_norm
        
        # Right vector (x-axis) - use a more robust method
        # Use world up vector [0, 0, 1] as reference
        world_up = np.array([0, 0, 1])
        right = np.cross(forward, world_up)
        right_norm = np.linalg.norm(right)
        
        # If forward is parallel to world_up, use a different reference
        if right_norm < 1e-6:
            world_right = np.array([1, 0, 0])
            right = np.cross(forward, world_right)
            right_norm = np.linalg.norm(right)
        
        if right_norm < 1e-6:
            # Fallback: use arbitrary perpendicular vector
            if abs(forward[2]) < 0.9:
                right = np.cross(forward, np.array([0, 0, 1]))
            else:
                right = np.cross(forward, np.array([1, 0, 0]))
            right = right / np.linalg.norm(right)
        else:
            right = right / right_norm
        
        # Down vector (y-axis) - ensure right-handed coordinate system
        down = np.cross(right, forward)
        down = down / np.linalg.norm(down)
        
        # Build rotation matrix (camera frame: x-right, y-down, z-forward)
        R_cam = np.column_stack([right, down, forward])
        
        # Ensure it's a valid rotation matrix (orthogonal, det=1)
        # Use SVD to fix if needed
        U, s, Vt = np.linalg.svd(R_cam)
        R_cam = U @ Vt
        if np.linalg.det(R_cam) < 0:
            U[:, -1] *= -1
            R_cam = U @ Vt
        
        # Build pose matrix (T_c2w)
        pose = np.eye(4)
        pose[:3, :3] = R_cam
        pose[:3, 3] = np.array([x, y, z])
        
        poses_gt.append(pose)
    
    return np.array(poses_gt)

def create_ground_truth_landmarks(num_landmarks, spread):
    """Create landmarks randomly distributed around origin."""
    landmarks_gt = np.random.uniform(
        -spread/2, spread/2, size=(num_landmarks, 3)
    )
    # Ensure landmarks are in front of cameras (positive z in world)
    landmarks_gt[:, 2] = np.abs(landmarks_gt[:, 2]) + 0.5
    return landmarks_gt

# Generate ground truth
poses_gt = create_ground_truth_trajectory(NUM_FRAMES, TRAJECTORY_RADIUS, TRAJECTORY_HEIGHT)
landmarks_gt = create_ground_truth_landmarks(NUM_LANDMARKS, LANDMARK_SPREAD)
landmark_ids = np.arange(NUM_LANDMARKS)

print(f"Generated {len(poses_gt)} ground truth poses")
print(f"Generated {len(landmarks_gt)} ground truth landmarks")
print(f"Landmark range: [{landmarks_gt.min(axis=0)}, {landmarks_gt.max(axis=0)}]")


In [ ]:
landmarks_gt[0]

In [ ]:
def add_noise_to_pose(pose, trans_noise, rot_noise):
    """Add noise to a pose matrix."""
    # Extract rotation and translation
    R_gt = pose[:3, :3]
    t_gt = pose[:3, 3]
    
    # Add translation noise
    t_noisy = t_gt + np.random.normal(0, trans_noise, size=3)
    
    # Add rotation noise (small angle approximation)
    rot_noise_vec = np.random.normal(0, rot_noise, size=3)
    R_noise = R.from_rotvec(rot_noise_vec).as_matrix()
    R_noisy = R_noise @ R_gt
    
    # Build noisy pose
    pose_noisy = np.eye(4)
    pose_noisy[:3, :3] = R_noisy
    pose_noisy[:3, 3] = t_noisy
    
    return pose_noisy

def compute_relative_pose(pose_prev, pose_curr):
    """Compute relative pose from prev to curr."""
    # rel_pose = transformation takes from prev to curr
    pose_curr_inv = inverse_SE3(pose_curr)
    rel_pose =  pose_curr_inv @ pose_prev
    return rel_pose

def project_landmark_to_camera(pose_c2w, landmark_world):
    """Project landmark from world to camera frame and return bearing-range."""
    # Transform landmark to camera frame
    pose_w2c = inverse_SE3(pose_c2w)
    landmark_cam = transform_pts(pose_w2c, landmark_world.reshape(1, -1))[0]
    
    # Compute range and bearing
    range_val = np.linalg.norm(landmark_cam)
    if range_val < 1e-9:
        return None, None
    
    bearing = landmark_cam / range_val  # Unit vector
    return bearing, range_val

# Generate noisy poses and observations
poses_noisy = []
rel_poses_gt = []
observations = []  # List of dicts: {frame_id: {landmark_id: (bearing, range)}}

for i in range(NUM_FRAMES):
    ## Pose_gt takes points from camera to world
    # Add noise to pose
    pose_noisy = add_noise_to_pose(
        poses_gt[i], POSE_NOISE_TRANSLATION, POSE_NOISE_ROTATION
    )
    poses_noisy.append(pose_noisy)
    
    # Compute relative pose
    if i == 0:
        rel_pose = np.eye(4)
    else:
        rel_pose = compute_relative_pose(poses_gt[i-1], poses_gt[i])

    # rel pose takes previous frame to current frame
    rel_poses_gt.append(rel_pose)
    
    # Generate observations (which landmarks are visible from this pose)
    frame_obs = {}
    for j, landmark in enumerate(landmarks_gt):
        bearing, range_val = project_landmark_to_camera(poses_gt[i], landmark)
        
        if bearing is not None and range_val > 0.1:  # Valid observation
            # Add noise to measurement
            bearing_noisy = bearing + np.random.normal(0, BEARING_NOISE, size=3)
            bearing_noisy = bearing_noisy / np.linalg.norm(bearing_noisy)
            range_noisy = range_val + np.random.normal(0, LANDMARK_MEASUREMENT_NOISE)
            range_noisy = max(0.1, range_noisy)  # Ensure positive
            
            frame_obs[j] = (bearing_noisy, range_noisy)
    
    observations.append(frame_obs)

# Convert to numpy arrays for easier indexing
poses_noisy = np.array(poses_noisy)
rel_poses_gt = np.array(rel_poses_gt)

print(f"Generated noisy poses and observations")
print(f"Average observations per frame: {np.mean([len(obs) for obs in observations]):.1f}")


In [ ]:
def create_object_frame_data(frame_id, pose_noisy, rel_pose, observations_frame, landmark_ids_all):
    """Create ObjectFrameData from observations."""
    # Collect observed landmarks for this frame
    cur_3d_list = []
    cur_3d_idx_list = []
    inliers_list = []
    residuals_list = []
    uncertainties_list = []
    
    for lid in observations_frame.keys():
        bearing, range_val = observations_frame[lid]
        # Convert bearing-range back to 3D point in camera frame
        point_cam = bearing * range_val
        cur_3d_list.append(point_cam)
        cur_3d_idx_list.append(lid)
        inliers_list.append(True)  # All are inliers in synthetic data
        residuals_list.append(LANDMARK_MEASUREMENT_NOISE)  # Approximate residual
        uncertainties_list.append(LANDMARK_MEASUREMENT_NOISE)
        
    

    cur_3d = np.array(cur_3d_list)
    cur_3d_idx = np.array(cur_3d_idx_list, dtype=np.int64)
    inliers = np.array(inliers_list, dtype=bool)
    residuals = np.array(residuals_list)
    uncertainties = np.array(uncertainties_list)
    
    return ObjectFrameData(
        obj_id=OBJ_ID,
        frame_id=frame_id,
        pose=pose_noisy,  # world_T_cam
        rel_pose=rel_pose,
        cur_3d=cur_3d,  # 3D points in camera frame
        cur_3d_idx=cur_3d_idx,
        inliers=inliers,
        residuals=residuals,
        uncertainties=uncertainties,
    )

# Create optimizer
optimizer_config = edict({
    "max_iterations": 500,
    "relative_error_tol": 1e-5,
    "absolute_error_tol": 1e-5,
    "prior_noise_param": [0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
})

optimizer = LMGraphOptimizer(optimizer_config)



# Run optimization frame by frame
optimizer_results = []
for i in range(NUM_FRAMES):
# for i in range(3):

    # pose_noisy takes from camera to world,
    # but our optmizer takes poses from world to camera, thus the inverse
    # rel_poses_gt takes from previous to current frame,
    data = create_object_frame_data(
        i, inverse_SE3(poses_noisy[i]), rel_poses_gt[i], observations[i], landmark_ids
    )

    result = optimizer.optimize(data)
    optimizer_results.append(result)

    print(optimizer._graph)
    print("--------------------------------")

print(f"Optimization complete for {NUM_FRAMES} frames")


In [ ]:
print(optimizer._graph)

## !!!! TODO: use graph.value directly to plot

In [ ]:
# import graphviz
# display(graphviz.Source(optimizer._graph.dot(optimizer._values)))

In [ ]:
# print(f"gt landmark: {landmarks_gt[28]}")
# f = optimizer._graph.at(588)
# br = f.measured()                 # BearingRange measurement
# bearing_u = br.bearing()          # Unit3
# range_val = float(br.range())     # scalar

# bearing_vec = np.asarray(bearing_u.unitVector(), dtype=float).reshape(3,)
# point_cam = bearing_vec * range_val
# print(f"bearing_vec: {bearing_vec}")
# print(f"range_val: {range_val}")
# print(f"point_cam: {point_cam}")

# # Use the pose that matches the factor key x4
# # Option A: from ground-truth array (if poses_gt[i] matches xi)
# point_world_gt = transform_pts(poses_gt[19], point_cam.reshape(1, -1))[0]
# print(f"point_world (using poses_gt[4]): {point_world_gt}")



In [ ]:
vals = optimizer._values  # or `result` after optimize
errs = []
for i in range(optimizer._graph.size()):
    f = optimizer._graph.at(i)
    errs.append((float(f.error(vals)), i, type(f).__name__, str(f.keys())))
errs.sort(reverse=True)
print("Top 10 factor errors:")
for e, i, t, ks in errs[:10]:
    print(i, t, ks, e)

In [ ]:
# fmt = gtsam.GraphvizFormatting()
# fmt.mergeSimilarFactors = True
# fmt.plotFactorPoints = True
# fmt.connectKeysToFactor = True
# fmt.binaryEdges = True

# # optimizer._graph.saveGraph("fg.dot", optimizer._values, fmt, gtsam.DefaultKeyFormatter)
# optimizer._graph.saveGraph("fg.dot", optimizer._values, gtsam.DefaultKeyFormatter, fmt)

In [ ]:
# Extract poses and landmarks directly from optimizer._values
# This visualizes the current state of the optimizer's internal graph

def extract_from_optimizer(optimizer):
    """Extract poses and landmarks directly from optimizer._values."""
    poses_from_optimizer = []
    landmarks_from_optimizer = {}
    
    # Extract all poses (symbols starting with 'x')
    for i in range(NUM_FRAMES):
        Xi = gtsam.symbol("x", i)
        if optimizer._values.exists(Xi):
            try:
                pose_c2w = optimizer._values.atPose3(Xi).matrix()
                # Convert from cam_T_world to world_T_cam for visualization
                # pose_w2c = inverse_SE3(pose_c2w.matrix())
                poses_from_optimizer.append(pose_c2w)
            except RuntimeError:
                poses_from_optimizer.append(None)
        else:
            poses_from_optimizer.append(None)
    
    # Extract all landmarks (symbols starting with 'l')
    # Get landmark IDs from optimizer
    if hasattr(optimizer, 'inserted_landmark_ids'):
        for lid in optimizer.inserted_landmark_ids:
            Lj = gtsam.symbol("l", int(lid))
            if optimizer._values.exists(Lj):
                try:
                    landmark_world = optimizer._values.atPoint3(Lj)
                    landmarks_from_optimizer[int(lid)] = np.array([
                        landmark_world, landmark_world, landmark_world
                    ])
                except RuntimeError:
                    pass
    
    return np.array(poses_from_optimizer), landmarks_from_optimizer

# Extract data from optimizer
poses_from_opt, landmarks_from_opt = extract_from_optimizer(optimizer)

print(f"Extracted {len([p for p in poses_from_opt if p is not None])} poses from optimizer")
print(f"Extracted {len(landmarks_from_opt)} landmarks from optimizer")


In [ ]:
# Visualization: 3D plot extracted directly from optimizer._values
fig = plt.figure(figsize=(10, 10))

# Ground truth
ax1 = fig.add_subplot(221, projection='3d')
ax1.set_title('Ground Truth', fontsize=14, fontweight='bold')

# Plot camera trajectory
for i, pose in enumerate(poses_gt):
    pos = pose[:3, 3]
    ax1.scatter(*pos, c='blue', s=50, alpha=0.6)
    if i == 0:
        ax1.scatter(*pos, c='blue', s=200, marker='*', label='Camera poses')
    
    # Draw camera frame
    scale = 0.1
    x_axis = pose[:3, 0] * scale
    y_axis = pose[:3, 1] * scale
    z_axis = pose[:3, 2] * scale
    ax1.quiver(*pos, *x_axis, color='r', arrow_length_ratio=0.3, alpha=0.5)
    ax1.quiver(*pos, *y_axis, color='g', arrow_length_ratio=0.3, alpha=0.5)
    ax1.quiver(*pos, *z_axis, color='b', arrow_length_ratio=0.3, alpha=0.5)

# Plot landmarks
ax1.scatter(*landmarks_gt.T, c='red', s=30, alpha=0.7, label='Landmarks')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.legend()
ax1.grid(True)

# Initial (noisy)
ax2 = fig.add_subplot(222, projection='3d')
ax2.set_title('Initial (Noisy)', fontsize=14, fontweight='bold')

for i, pose in enumerate(poses_noisy):
    pos = pose[:3, 3]
    ax2.scatter(*pos, c='orange', s=50, alpha=0.6)
    if i == 0:
        ax2.scatter(*pos, c='orange', s=200, marker='*', label='Camera poses')

# Plot observed landmarks (reconstructed from noisy measurements)
landmarks_initial = []
for i, obs in enumerate(observations):
    for lid, (bearing, range_val) in obs.items():
        point_cam = bearing * range_val
        # Transform to world using noisy pose
        point_world = transform_pts(poses_noisy[i], point_cam.reshape(1, -1))[0]
        landmarks_initial.append(point_world)

if landmarks_initial:
    landmarks_initial = np.array(landmarks_initial)
    ax2.scatter(*landmarks_initial.T, c='pink', s=20, alpha=0.5, label='Observed landmarks')

ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_zlabel('Z (m)')
ax2.legend()
ax2.grid(True)

# Optimized (from optimizer._values)
ax3 = fig.add_subplot(223, projection='3d')
ax3.set_title('Optimized (from optimizer._values)', fontsize=14, fontweight='bold')

for i, pose in enumerate(poses_from_opt):
    if pose is not None:
        pos = pose[:3, 3]
        ax3.scatter(*pos, c='green', s=50, alpha=0.6)
        if i == 0:
            ax3.scatter(*pos, c='green', s=200, marker='*', label='Camera poses')
        
        # Draw camera frame
        scale = 0.1
        x_axis = pose[:3, 0] * scale
        y_axis = pose[:3, 1] * scale
        z_axis = pose[:3, 2] * scale
        ax3.quiver(*pos, *x_axis, color='r', arrow_length_ratio=0.3, alpha=0.5)
        ax3.quiver(*pos, *y_axis, color='g', arrow_length_ratio=0.3, alpha=0.5)
        ax3.quiver(*pos, *z_axis, color='b', arrow_length_ratio=0.3, alpha=0.5)

if landmarks_from_opt:
    landmark_positions = np.array([landmarks_from_opt[lid] for lid in sorted(landmarks_from_opt.keys())])
    ax3.scatter(*landmark_positions.T, c='purple', s=30, alpha=0.7, label='Landmarks')

ax3.set_xlabel('X (m)')
ax3.set_ylabel('Y (m)')
ax3.set_zlabel('Z (m)')
ax3.legend()
ax3.grid(True)

# Comparison: Ground truth vs Optimized
ax4 = fig.add_subplot(224, projection='3d')
ax4.set_title('GT vs Optimized (from optimizer)', fontsize=14, fontweight='bold')

# Plot ground truth trajectory
for i, pose in enumerate(poses_gt):
    pos = pose[:3, 3]
    ax4.scatter(*pos, c='blue', s=50, alpha=0.6, marker='o')
    if i == 0:
        ax4.scatter(*pos, c='blue', s=200, marker='*', label='GT Camera poses')

# Plot optimized trajectory
for i, pose in enumerate(poses_from_opt):
    if pose is not None:
        pos = pose[:3, 3]
        ax4.scatter(*pos, c='green', s=50, alpha=0.6, marker='^')
        if i == 0:
            ax4.scatter(*pos, c='green', s=200, marker='*', label='Optimized Camera poses')
        # Draw line connecting GT to optimized
        if i < len(poses_gt):
            pos_gt = poses_gt[i][:3, 3]
            ax4.plot([pos_gt[0], pos[0]], [pos_gt[1], pos[1]], [pos_gt[2], pos[2]], 
                    'r--', alpha=0.3, linewidth=0.5)

# Plot landmarks
ax4.scatter(*landmarks_gt.T, c='red', s=30, alpha=0.5, marker='o', label='GT Landmarks')
if landmarks_from_opt:
    landmark_positions = np.array([landmarks_from_opt[lid] for lid in sorted(landmarks_from_opt.keys())])
    ax4.scatter(*landmark_positions.T, c='purple', s=30, alpha=0.7, marker='^', label='Optimized Landmarks')

ax4.set_xlabel('X (m)')
ax4.set_ylabel('Y (m)')
ax4.set_zlabel('Z (m)')
ax4.legend()
ax4.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Visualize factors before and after optimization
# Shows pose-pose (between) and pose-landmark (bearing-range) factors from optimizer._graph

def plot_factor_graph(ax, optimizer, values_to_use, title, use_initial=False):
    """Plot factor graph with given values."""
    # Extract pose and landmark positions
    pose_positions = {}
    landmark_positions = {}
    
    # Poses
    for i in range(NUM_FRAMES):
        Xi = gtsam.symbol('x', i)
        if values_to_use.exists(Xi):
            try:
                if use_initial:
                    # Use initial noisy poses
                    p = poses_noisy[i][:3, 3]
                    pose_positions[Xi] = p
                else:
                    # Use optimized poses from values
                    pose_c2w = values_to_use.atPose3(Xi).matrix()
                    # pose_world_T_cam = inverse_SE3(pose_cam_T_world.matrix())
                    p = pose_c2w[:3, 3]
                    pose_positions[Xi] = p
                ax.scatter(*p, c='blue', s=40, alpha=0.8, marker='o')
            except RuntimeError:
                pass
    
    # Landmarks
    if hasattr(optimizer, 'inserted_landmark_ids'):
        for lid in optimizer.inserted_landmark_ids:
            Lj = gtsam.symbol('l', int(lid))
            if values_to_use.exists(Lj):
                try:
                    if use_initial:
                        # For initial, reconstruct landmark from first observation
                        # Find first frame that observed this landmark
                        for frame_idx, obs in enumerate(observations):
                            if lid in obs:
                                bearing, range_val = obs[lid]
                                point_cam = bearing * range_val
                                point_world = transform_pts(poses_noisy[frame_idx], point_cam.reshape(1, -1))[0]
                                p = point_world
                                landmark_positions[Lj] = p
                                break
                    else:
                        # Use optimized landmarks from values
                        p = values_to_use.atPoint3(Lj)
                        landmark_positions[Lj] = p
                    ax.scatter(*p, c='red', s=25, alpha=0.8, marker='^')
                except RuntimeError:
                    pass
    
    # Draw factors as edges
    num_factors = optimizer._graph.size()
    between_count = 0
    bearing_count = 0
    
    for k in range(num_factors):
        f = optimizer._graph.at(k)
        # BetweenFactorPose3: pose-pose edge
        if isinstance(f, gtsam.BetweenFactorPose3):
            keys = f.keys()
            if len(keys) == 2:
                k1, k2 = keys[0], keys[1]
                if k1 in pose_positions and k2 in pose_positions:
                    p1 = pose_positions[k1]
                    p2 = pose_positions[k2]
                    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                            color='blue', alpha=0.4, linewidth=1.5)
                    between_count += 1
        # BearingRangeFactor3D: pose-landmark edge
        elif isinstance(f, gtsam.BearingRangeFactor3D):
            keys = f.keys()
            if len(keys) == 2:
                k_pose, k_land = keys[0], keys[1]
                if k_pose in pose_positions and k_land in landmark_positions:
                    p1 = pose_positions[k_pose]
                    p2 = landmark_positions[k_land]
                    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]],
                            color='green', alpha=0.3, linewidth=0.8)
                    bearing_count += 1
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(f'{title}\n({len(pose_positions)} poses, {len(landmark_positions)} landmarks, '
                 f'{between_count} between factors, {bearing_count} bearing-range factors)',
                 fontsize=12, fontweight='bold')
    ax.grid(True)
    ax.view_init(elev=30, azim=-60)

# Create figure with two subplots
fig = plt.figure(figsize=(16, 8))

# Before optimization (using initial noisy values)
ax1 = fig.add_subplot(121, projection='3d')
# Create a temporary Values object with initial poses
initial_values = gtsam.Values()
for i in range(NUM_FRAMES):
    Xi = gtsam.symbol('x', i)
    pose_cam_T_world = gtsam.Pose3(inverse_SE3(poses_noisy[i]))
    initial_values.insert(Xi, pose_cam_T_world)
# Add initial landmarks (reconstructed from first observation)
if hasattr(optimizer, 'inserted_landmark_ids'):
    for lid in optimizer.inserted_landmark_ids:
        Lj = gtsam.symbol('l', int(lid))
        # Find first observation
        for frame_idx, obs in enumerate(observations):
            if lid in obs:
                bearing, range_val = obs[lid]
                point_cam = bearing * range_val
                point_world = transform_pts(poses_noisy[frame_idx], point_cam.reshape(1, -1))[0]
                initial_values.insert(Lj, gtsam.Point3(*point_world))
                break

plot_factor_graph(ax1, optimizer, initial_values, 'Before Optimization', use_initial=True)

# After optimization (using optimized values)
ax2 = fig.add_subplot(122, projection='3d')
plot_factor_graph(ax2, optimizer, optimizer._values, 'After Optimization', use_initial=False)

plt.tight_layout()
plt.show()


In [ ]:
errs = []
for i in range(optimizer._graph.size()):
    f = optimizer._graph.at(i)
    e = float(f.error(optimizer._values))   # or result after optimize
    errs.append((e, i, type(f).__name__))

print(errs)

In [ ]:
# Collect optimized results
poses_optimized = []
landmarks_optimized_dict = {}  # landmark_id -> position

for i, result in enumerate(optimizer_results):
    if result is not None:
        # result.pose_optimized takes from camera to world
        # but we want poses from world to camera
        poses_optimized.append(inverse_SE3(result.pose_optimized)  )
        
        # Store optimized landmarks
        for j, lid in enumerate(result.key_points_idx_optimized):
            landmarks_optimized_dict[int(lid)] = result.key_points_optimized[j]
    else:
        # Use noisy pose if optimization failed
        poses_optimized.append(poses_noisy[i])

poses_optimized = np.array(poses_optimized)

# Convert landmarks dict to array (matching ground truth order)
landmarks_optimized = np.array([
    landmarks_optimized_dict.get(lid, np.array([np.nan, np.nan, np.nan]))
    for lid in landmark_ids
])

# Filter out NaN landmarks
valid_mask = ~np.isnan(landmarks_optimized).any(axis=1)
landmarks_optimized_valid = landmarks_optimized[valid_mask]
landmark_ids_valid = landmark_ids[valid_mask]

print(f"Optimized {len(poses_optimized)} poses")
print(f"Optimized {len(landmarks_optimized_valid)} landmarks (out of {NUM_LANDMARKS})")


In [ ]:
# Visualization: 3D plot of ground truth, initial, and optimized
fig = plt.figure(figsize=(10, 10))

# Ground truth
ax1 = fig.add_subplot(221, projection='3d')
ax1.set_title('Ground Truth', fontsize=14, fontweight='bold')

# Plot camera trajectory
for i, pose in enumerate(poses_gt):
    pos = pose[:3, 3]
    ax1.scatter(*pos, c='blue', s=50, alpha=0.6)
    if i == 0:
        ax1.scatter(*pos, c='blue', s=200, marker='*', label='Camera poses')
    
    # Draw camera frame
    scale = 0.1
    x_axis = pose[:3, 0] * scale
    y_axis = pose[:3, 1] * scale
    z_axis = pose[:3, 2] * scale
    ax1.quiver(*pos, *x_axis, color='r', arrow_length_ratio=0.3, alpha=0.5)
    ax1.quiver(*pos, *y_axis, color='g', arrow_length_ratio=0.3, alpha=0.5)
    ax1.quiver(*pos, *z_axis, color='b', arrow_length_ratio=0.3, alpha=0.5)

# Plot landmarks
ax1.scatter(*landmarks_gt.T, c='red', s=30, alpha=0.7, label='Landmarks')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.legend()
ax1.grid(True)

# Initial (noisy)
ax2 = fig.add_subplot(222, projection='3d')
ax2.set_title('Initial (Noisy)', fontsize=14, fontweight='bold')

for i, pose in enumerate(poses_noisy):
    pos = pose[:3, 3]
    ax2.scatter(*pos, c='orange', s=50, alpha=0.6)
    if i == 0:
        ax2.scatter(*pos, c='orange', s=200, marker='*', label='Camera poses')

# Plot observed landmarks (reconstructed from noisy measurements)
landmarks_initial = []
for i, obs in enumerate(observations):
    for lid, (bearing, range_val) in obs.items():
        point_cam = bearing * range_val
        # Transform to world using noisy pose
        point_world = transform_pts(poses_noisy[i], point_cam.reshape(1, -1))[0]
        landmarks_initial.append(point_world)

if landmarks_initial:
    landmarks_initial = np.array(landmarks_initial)
    ax2.scatter(*landmarks_initial.T, c='pink', s=20, alpha=0.5, label='Observed landmarks')

ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_zlabel('Z (m)')
ax2.legend()
ax2.grid(True)

# Optimized
ax3 = fig.add_subplot(223, projection='3d')
ax3.set_title('Optimized', fontsize=14, fontweight='bold')

for i, pose in enumerate(poses_optimized):
    pos = pose[:3, 3]
    ax3.scatter(*pos, c='green', s=50, alpha=0.6)
    if i == 0:
        ax3.scatter(*pos, c='green', s=200, marker='*', label='Camera poses')
    
    # Draw camera frame
    scale = 0.1
    x_axis = pose[:3, 0] * scale
    y_axis = pose[:3, 1] * scale
    z_axis = pose[:3, 2] * scale
    ax3.quiver(*pos, *x_axis, color='r', arrow_length_ratio=0.3, alpha=0.5)
    ax3.quiver(*pos, *y_axis, color='g', arrow_length_ratio=0.3, alpha=0.5)
    ax3.quiver(*pos, *z_axis, color='b', arrow_length_ratio=0.3, alpha=0.5)

if len(landmarks_optimized_valid) > 0:
    ax3.scatter(*landmarks_optimized_valid.T, c='purple', s=30, alpha=0.7, label='Landmarks')

ax3.set_xlabel('X (m)')
ax3.set_ylabel('Y (m)')
ax3.set_zlabel('Z (m)')
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Analysis: Pose errors
def compute_pose_error(pose1, pose2):
    """Compute translation and rotation errors between two poses."""
    # Translation error
    t1 = pose1[:3, 3]
    t2 = pose2[:3, 3]
    trans_error = np.linalg.norm(t1 - t2)
    
    # Rotation error (angle of relative rotation)
    R1 = pose1[:3, :3]
    R2 = pose2[:3, :3]
    R_rel = R1.T @ R2
    rot_error = np.arccos(np.clip((np.trace(R_rel) - 1) / 2, -1, 1))
    
    return trans_error, rot_error

# Compute errors
pose_errors_initial = []
pose_errors_optimized = []

for i in range(NUM_FRAMES):
    # Initial error
    trans_err_init, rot_err_init = compute_pose_error(poses_gt[i], poses_noisy[i])
    pose_errors_initial.append((trans_err_init, rot_err_init))
    
    # Optimized error
    if i < len(poses_optimized):
        trans_err_opt, rot_err_opt = compute_pose_error(poses_gt[i], poses_optimized[i])
        pose_errors_optimized.append((trans_err_opt, rot_err_opt))
    else:
        pose_errors_optimized.append((np.nan, np.nan))

pose_errors_initial = np.array(pose_errors_initial)
pose_errors_optimized = np.array(pose_errors_optimized)

# Landmark errors
landmark_errors_optimized = []
for lid in landmark_ids_valid:
    if lid < len(landmarks_gt):
        error = np.linalg.norm(landmarks_gt[lid] - landmarks_optimized_dict[int(lid)])
        landmark_errors_optimized.append(error)

landmark_errors_optimized = np.array(landmark_errors_optimized)

# Plot analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Pose translation errors
ax = axes[0, 0]
ax.plot(pose_errors_initial[:, 0], 'o-', label='Initial (noisy)', alpha=0.7)
ax.plot(pose_errors_optimized[:, 0], 's-', label='Optimized', alpha=0.7)
ax.set_xlabel('Frame ID')
ax.set_ylabel('Translation Error (m)')
ax.set_title('Pose Translation Errors')
ax.legend()
ax.grid(True, alpha=0.3)

# Pose rotation errors
ax = axes[0, 1]
ax.plot(pose_errors_initial[:, 1], 'o-', label='Initial (noisy)', alpha=0.7)
ax.plot(pose_errors_optimized[:, 1], 's-', label='Optimized', alpha=0.7)
ax.set_xlabel('Frame ID')
ax.set_ylabel('Rotation Error (rad)')
ax.set_title('Pose Rotation Errors')
ax.legend()
ax.grid(True, alpha=0.3)

# Landmark errors
ax = axes[1, 0]
if len(landmark_errors_optimized) > 0:
    ax.hist(landmark_errors_optimized, bins=20, alpha=0.7, edgecolor='black')
    ax.axvline(landmark_errors_optimized.mean(), color='r', linestyle='--', 
               label=f'Mean: {landmark_errors_optimized.mean():.4f} m')
    ax.set_xlabel('Landmark Position Error (m)')
    ax.set_ylabel('Frequency')
    ax.set_title('Landmark Position Errors (Optimized)')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No landmarks optimized', ha='center', va='center')
    ax.set_title('Landmark Position Errors')

# Error statistics
ax = axes[1, 1]
ax.axis('off')
stats_text = f"""
Error Statistics:

Pose Translation Errors:
  Initial - Mean: {pose_errors_initial[:, 0].mean():.4f} m, Std: {pose_errors_initial[:, 0].std():.4f} m
  Optimized - Mean: {pose_errors_optimized[:, 0].mean():.4f} m, Std: {pose_errors_optimized[:, 0].std():.4f} m
  Improvement: {(1 - pose_errors_optimized[:, 0].mean() / pose_errors_initial[:, 0].mean()) * 100:.1f}%

Pose Rotation Errors:
  Initial - Mean: {pose_errors_initial[:, 1].mean():.4f} rad, Std: {pose_errors_initial[:, 1].std():.4f} rad
  Optimized - Mean: {pose_errors_optimized[:, 1].mean():.4f} rad, Std: {pose_errors_optimized[:, 1].std():.4f} rad
  Improvement: {(1 - pose_errors_optimized[:, 1].mean() / pose_errors_initial[:, 1].mean()) * 100:.1f}%

Landmark Position Errors:
  Optimized - Mean: {landmark_errors_optimized.mean():.4f} m, Std: {landmark_errors_optimized.std():.4f} m
  (out of {len(landmark_errors_optimized)} optimized landmarks)
"""
ax.text(0.1, 0.5, stats_text, fontsize=10, family='monospace', verticalalignment='center')

plt.tight_layout()
plt.show()


In [ ]:
# Analysis: Trajectory comparison
fig = plt.figure(figsize=(16, 5))

# X-Y trajectory
ax1 = fig.add_subplot(131)
ax1.plot(poses_gt[:, 0, 3], poses_gt[:, 1, 3], 'b-o', label='Ground Truth', alpha=0.7, markersize=6)
ax1.plot(poses_noisy[:, 0, 3], poses_noisy[:, 1, 3], 'orange', marker='s', linestyle='--', 
         label='Initial (Noisy)', alpha=0.7, markersize=4)
ax1.plot(poses_optimized[:, 0, 3], poses_optimized[:, 1, 3], 'g-^', label='Optimized', 
         alpha=0.7, markersize=4)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('Trajectory (X-Y view)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# X-Z trajectory
ax2 = fig.add_subplot(132)
ax2.plot(poses_gt[:, 0, 3], poses_gt[:, 2, 3], 'b-o', label='Ground Truth', alpha=0.7, markersize=6)
ax2.plot(poses_noisy[:, 0, 3], poses_noisy[:, 2, 3], 'orange', marker='s', linestyle='--', 
         label='Initial (Noisy)', alpha=0.7, markersize=4)
ax2.plot(poses_optimized[:, 0, 3], poses_optimized[:, 2, 3], 'g-^', label='Optimized', 
         alpha=0.7, markersize=4)
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Z (m)')
ax2.set_title('Trajectory (X-Z view)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# Y-Z trajectory
ax3 = fig.add_subplot(133)
ax3.plot(poses_gt[:, 1, 3], poses_gt[:, 2, 3], 'b-o', label='Ground Truth', alpha=0.7, markersize=6)
ax3.plot(poses_noisy[:, 1, 3], poses_noisy[:, 2, 3], 'orange', marker='s', linestyle='--', 
         label='Initial (Noisy)', alpha=0.7, markersize=4)
ax3.plot(poses_optimized[:, 1, 3], poses_optimized[:, 2, 3], 'g-^', label='Optimized', 
         alpha=0.7, markersize=4)
ax3.set_xlabel('Y (m)')
ax3.set_ylabel('Z (m)')
ax3.set_title('Trajectory (Y-Z view)')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()


In [ ]:
# Analysis: Landmark comparison
fig = plt.figure(figsize=(16, 5))

# Prepare data: only show landmarks that were optimized
if len(landmarks_optimized_valid) > 0:
    landmarks_gt_matched = landmarks_gt[landmark_ids_valid]
else:
    landmarks_gt_matched = np.empty((0, 3))
    landmarks_optimized_valid = np.empty((0, 3))

# X-Y view
ax1 = fig.add_subplot(131)
if len(landmarks_gt_matched) > 0:
    ax1.scatter(landmarks_gt_matched[:, 0], landmarks_gt_matched[:, 1], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o')
    ax1.scatter(landmarks_optimized_valid[:, 0], landmarks_optimized_valid[:, 1], 
               c='purple', s=30, alpha=0.7, label='Optimized', marker='^')
    # Draw lines connecting GT to optimized
    for i, lid in enumerate(landmark_ids_valid):
        if lid < len(landmarks_gt):
            gt_pos = landmarks_gt_matched[i]
            opt_pos = landmarks_optimized_valid[i]
            ax1.plot([gt_pos[0], opt_pos[0]], [gt_pos[1], opt_pos[1]], 
                    'k--', alpha=0.3, linewidth=0.5)
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('Landmarks (X-Y view)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# X-Z view
ax2 = fig.add_subplot(132)
if len(landmarks_gt_matched) > 0:
    ax2.scatter(landmarks_gt_matched[:, 0], landmarks_gt_matched[:, 2], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o')
    ax2.scatter(landmarks_optimized_valid[:, 0], landmarks_optimized_valid[:, 2], 
               c='purple', s=30, alpha=0.7, label='Optimized', marker='^')
    for i, lid in enumerate(landmark_ids_valid):
        if lid < len(landmarks_gt):
            gt_pos = landmarks_gt_matched[i]
            opt_pos = landmarks_optimized_valid[i]
            ax2.plot([gt_pos[0], opt_pos[0]], [gt_pos[2], opt_pos[2]], 
                    'k--', alpha=0.3, linewidth=0.5)
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Z (m)')
ax2.set_title('Landmarks (X-Z view)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# Y-Z view
ax3 = fig.add_subplot(133)
if len(landmarks_gt_matched) > 0:
    ax3.scatter(landmarks_gt_matched[:, 1], landmarks_gt_matched[:, 2], c='red', s=50, alpha=0.7, 
               label='Ground Truth', marker='o')
    ax3.scatter(landmarks_optimized_valid[:, 1], landmarks_optimized_valid[:, 2], 
               c='purple', s=30, alpha=0.7, label='Optimized', marker='^')
    for i, lid in enumerate(landmark_ids_valid):
        if lid < len(landmarks_gt):
            gt_pos = landmarks_gt_matched[i]
            opt_pos = landmarks_optimized_valid[i]
            ax3.plot([gt_pos[1], opt_pos[1]], [gt_pos[2], opt_pos[2]], 
                    'k--', alpha=0.3, linewidth=0.5)
ax3.set_xlabel('Y (m)')
ax3.set_ylabel('Z (m)')
ax3.set_title('Landmarks (Y-Z view)')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

plt.tight_layout()
plt.show()


In [ ]:
# Analysis: Observation statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Number of observations per frame
ax = axes[0]
num_obs_per_frame = [len(obs) for obs in observations]
ax.plot(num_obs_per_frame, 'o-', alpha=0.7)
ax.set_xlabel('Frame ID')
ax.set_ylabel('Number of Observations')
ax.set_title('Observations per Frame')
ax.grid(True, alpha=0.3)

# Number of observations per landmark
ax = axes[1]
landmark_obs_count = np.zeros(NUM_LANDMARKS)
for obs in observations:
    for lid in obs.keys():
        landmark_obs_count[lid] += 1
ax.bar(range(NUM_LANDMARKS), landmark_obs_count, alpha=0.7)
ax.set_xlabel('Landmark ID')
ax.set_ylabel('Number of Observations')
ax.set_title('Observations per Landmark')
ax.grid(True, alpha=0.3)

# Range distribution
ax = axes[2]
ranges = []
for obs in observations:
    for bearing, range_val in obs.values():
        ranges.append(range_val)
ranges = np.array(ranges)
ax.hist(ranges, bins=20, alpha=0.7, edgecolor='black')
ax.set_xlabel('Range (m)')
ax.set_ylabel('Frequency')
ax.set_title('Range Distribution')
ax.axvline(ranges.mean(), color='r', linestyle='--', 
           label=f'Mean: {ranges.mean():.2f} m')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Detailed error analysis: per-component breakdown
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Translation errors (X, Y, Z components)
ax = axes[0, 0]
trans_errors_initial = poses_noisy[:, :3, 3] - poses_gt[:, :3, 3]
trans_errors_optimized = poses_optimized[:, :3, 3] - poses_gt[:, :3, 3]
ax.plot(trans_errors_initial[:, 0], 'o-', label='X (initial)', alpha=0.7)
ax.plot(trans_errors_initial[:, 1], 's-', label='Y (initial)', alpha=0.7)
ax.plot(trans_errors_initial[:, 2], '^-', label='Z (initial)', alpha=0.7)
ax.plot(trans_errors_optimized[:, 0], 'o--', label='X (optimized)', alpha=0.7)
ax.plot(trans_errors_optimized[:, 1], 's--', label='Y (optimized)', alpha=0.7)
ax.plot(trans_errors_optimized[:, 2], '^--', label='Z (optimized)', alpha=0.7)
ax.set_xlabel('Frame ID')
ax.set_ylabel('Translation Error (m)')
ax.set_title('Translation Errors (per component)')
ax.legend()
ax.grid(True, alpha=0.3)

# Rotation errors (Euler angles)
ax = axes[0, 1]
def pose_to_euler(pose):
    R_mat = pose[:3, :3]
    # Ensure valid rotation matrix
    U, s, Vt = np.linalg.svd(R_mat)
    R_mat_fixed = U @ Vt
    if np.linalg.det(R_mat_fixed) < 0:
        U[:, -1] *= -1
        R_mat_fixed = U @ Vt
    try:
        return R.from_matrix(R_mat_fixed).as_euler('xyz')
    except ValueError:
        # Fallback: return zeros if still invalid
        return np.array([0.0, 0.0, 0.0])

euler_gt = np.array([pose_to_euler(p) for p in poses_gt])
euler_initial = np.array([pose_to_euler(p) for p in poses_noisy])
euler_optimized = np.array([pose_to_euler(p) for p in poses_optimized])
rot_errors_initial = euler_initial - euler_gt
rot_errors_optimized = euler_optimized - euler_gt
# Wrap angles to [-pi, pi]
rot_errors_initial = np.arctan2(np.sin(rot_errors_initial), np.cos(rot_errors_initial))
rot_errors_optimized = np.arctan2(np.sin(rot_errors_optimized), np.cos(rot_errors_optimized))

ax.plot(rot_errors_initial[:, 0], 'o-', label='Roll (initial)', alpha=0.7)
ax.plot(rot_errors_initial[:, 1], 's-', label='Pitch (initial)', alpha=0.7)
ax.plot(rot_errors_initial[:, 2], '^-', label='Yaw (initial)', alpha=0.7)
ax.plot(rot_errors_optimized[:, 0], 'o--', label='Roll (optimized)', alpha=0.7)
ax.plot(rot_errors_optimized[:, 1], 's--', label='Pitch (optimized)', alpha=0.7)
ax.plot(rot_errors_optimized[:, 2], '^--', label='Yaw (optimized)', alpha=0.7)
ax.set_xlabel('Frame ID')
ax.set_ylabel('Rotation Error (rad)')
ax.set_title('Rotation Errors (Euler angles)')
ax.legend()
ax.grid(True, alpha=0.3)

# Landmark errors (per component)
ax = axes[0, 2]
if len(landmark_errors_optimized) > 0:
    landmark_errors_x = []
    landmark_errors_y = []
    landmark_errors_z = []
    for lid in landmark_ids_valid:
        if lid < len(landmarks_gt):
            error = landmarks_optimized_dict[int(lid)] - landmarks_gt[lid]
            landmark_errors_x.append(error[0])
            landmark_errors_y.append(error[1])
            landmark_errors_z.append(error[2])
    
    ax.scatter(range(len(landmark_errors_x)), landmark_errors_x, 
              label='X error', alpha=0.6, s=20)
    ax.scatter(range(len(landmark_errors_y)), landmark_errors_y, 
              label='Y error', alpha=0.6, s=20)
    ax.scatter(range(len(landmark_errors_z)), landmark_errors_z, 
              label='Z error', alpha=0.6, s=20)
    ax.set_xlabel('Landmark Index')
    ax.set_ylabel('Position Error (m)')
    ax.set_title('Landmark Position Errors (per component)')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No landmarks optimized', ha='center', va='center')
    ax.set_title('Landmark Position Errors')

# Cumulative error distribution
ax = axes[1, 0]
ax.hist(pose_errors_initial[:, 0], bins=20, alpha=0.5, label='Initial', edgecolor='black')
ax.hist(pose_errors_optimized[:, 0], bins=20, alpha=0.5, label='Optimized', edgecolor='black')
ax.set_xlabel('Translation Error (m)')
ax.set_ylabel('Frequency')
ax.set_title('Pose Translation Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.hist(pose_errors_initial[:, 1], bins=20, alpha=0.5, label='Initial', edgecolor='black')
ax.hist(pose_errors_optimized[:, 1], bins=20, alpha=0.5, label='Optimized', edgecolor='black')
ax.set_xlabel('Rotation Error (rad)')
ax.set_ylabel('Frequency')
ax.set_title('Pose Rotation Error Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Error vs frame (showing convergence)
ax = axes[1, 2]
ax.semilogy(pose_errors_initial[:, 0], 'o-', label='Translation (initial)', alpha=0.7)
ax.semilogy(pose_errors_optimized[:, 0], 's-', label='Translation (optimized)', alpha=0.7)
ax.semilogy(pose_errors_initial[:, 1] * 10, '^-', label='Rotation x10 (initial)', alpha=0.7)
ax.semilogy(pose_errors_optimized[:, 1] * 10, 'v-', label='Rotation x10 (optimized)', alpha=0.7)
ax.set_xlabel('Frame ID')
ax.set_ylabel('Error (log scale)')
ax.set_title('Error Convergence')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
